# CIFAR-10 small-ViT optimizer baselines

This notebook runs the matched **SGD + Nesterov**, **AdamW**, and **Muon + auxiliary AdamW** controls. The official CIFAR-10 training set is split once into 45,000 optimization examples and 5,000 validation examples. The official test set is monitoring-only; validation loss selects the best checkpoint.

Every uncertainty band is a two-sided **95% Student-t interval across three complete runs**. Transformer blocks and WeightWatcher fits are repeated measurements inside a run and are never counted as independent replicates. Layer plots therefore use a separate curve for every physical matrix in every block.

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

ROOT = None
for path in [Path.cwd(), *Path.cwd().parents]:
    candidate = path / 'baseline'
    if (candidate / 'rg_baselines').is_dir():
        ROOT = candidate
        break
    if (path / 'rg_baselines').is_dir():
        ROOT = path
        break
if ROOT is None:
    raise RuntimeError('Run this notebook from a clone of CalculatedContent/rg_optimizers.')
ROOT = ROOT.resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from rg_baselines.vit_runtime import (
    DEFAULT_VIT_SEEDS,
    ViTBaselineConfig,
    choose_device,
    run_vit_baseline,
)
from rg_baselines.vit_analysis import (
    OPTIMIZER_ORDER,
    load_vit_results,
    plot_layer_metric,
    plot_performance_metric,
    summarize_layer_metric,
    terminal_summary,
)

def resolved_path(variable, default):
    raw = os.environ.get(variable)
    path = Path(raw).expanduser() if raw else default
    return path.resolve()

DATA_DIR = resolved_path('RG_BASELINE_DATA_DIR', ROOT / 'data')
RUN_ROOT = resolved_path('RG_BASELINE_RUN_ROOT', ROOT / 'runs') / 'cifar10_vit'
PLOT_DIR = RUN_ROOT / 'plots'
DATA_DIR.mkdir(parents=True, exist_ok=True)
RUN_ROOT.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = choose_device()
CONFIG = ViTBaselineConfig()
SEEDS = DEFAULT_VIT_SEEDS
assert len(SEEDS) == 3 and len(set(SEEDS)) == 3
print('device:', DEVICE)
print('data:', DATA_DIR)
print('runs:', RUN_ROOT)
display(pd.DataFrame([CONFIG.__dict__]))

## Run or resume the nine matched jobs

Each run writes `checkpoint_latest.pt`, `checkpoint_best.pt`, periodic checkpoints, a protocol fingerprint, direct WeightWatcher outputs from `ERG=True, randomize=True`, and a completion marker. Rerunning this cell resumes compatible incomplete jobs and skips completed jobs. Randomized WeightWatcher measurements preserve every training RNG stream.

In [ ]:
for optimizer in OPTIMIZER_ORDER:
    for seed in SEEDS:
        run_vit_baseline(
            optimizer,
            seed,
            data_dir=DATA_DIR,
            output_dir=RUN_ROOT,
            config=CONFIG,
            device=DEVICE,
            progress=True,
            resume=True,
        )

history, spectral = load_vit_results(RUN_ROOT, seeds=SEEDS)
history.to_csv(RUN_ROOT / 'performance_all_runs.csv', index=False)
spectral.to_csv(RUN_ROOT / 'weightwatcher_all_runs.csv', index=False)
display(history.groupby(['optimizer', 'seed']).tail(1))

## Final and validation-selected results

The table reports both the final epoch and the checkpoint selected strictly by validation loss. Test values are reported only after selection.

In [ ]:
terminal = terminal_summary(history, expected_seeds=SEEDS)
terminal.to_csv(RUN_ROOT / 'terminal_summary_95ci.csv', index=False)
display(terminal.sort_values(['metric', 'checkpoint', 'optimizer']))

## Performance and learning-rate trajectories

Faint lines are individual runs; the heavy curve and band are the mean and 95% Student-t interval across the three run seeds.

In [ ]:
for metric in [
    'train_loss', 'validation_loss', 'test_loss',
    'train_accuracy', 'validation_accuracy', 'test_accuracy',
    'primary_lr', 'auxiliary_lr',
]:
    if metric not in history.columns or history[metric].notna().sum() == 0:
        continue
    plot_performance_metric(
        history, metric,
        output=PLOT_DIR / f'{metric}_95ci.png',
        expected_seeds=SEEDS,
    )
    plt.show()

## Layer-resolved WeightWatcher trajectories

`alpha`, `ERG_gap`, and `num_traps` are retained directly from WeightWatcher. Each figure is one transformer block, and each uncertainty interval contains exactly the three complete runs for the same physical matrix.

In [ ]:
for metric in ['alpha', 'ERG_gap', 'num_traps']:
    layer_summary = summarize_layer_metric(
        spectral, metric, expected_seeds=SEEDS
    )
    assert layer_summary['n'].eq(3).all()
    layer_summary.to_csv(
        RUN_ROOT / f'{metric}_by_matrix_95ci.csv', index=False
    )
    for optimizer in OPTIMIZER_ORDER:
        plot_layer_metric(
            spectral,
            optimizer=optimizer,
            metric=metric,
            output_dir=PLOT_DIR / 'layers',
            expected_seeds=SEEDS,
        )
        plt.show()
display(layer_summary.head(30))

## Interpretation contract

These settings are strong source-backed reference recipes. They are not presented as a mathematically proven global optimum. Any future hyperparameter sweep must use the optimization/validation split only and must be frozen before the protected test results are interpreted.